## 0. Ringkasan Improvement (exp_002)

Notebook ini adalah revisi dari `stage_b.ipynb` (exp_001). Hasil exp_001: **val macro-F1 mentok di 0.4863** (epoch 8), lalu stagnan/turun sampai early-stop di epoch 11, sementara train macro-F1 terus naik ke 0.55 — indikasi **overfitting** dan kapasitas head yang kurang optimal untuk memanfaatkan backbone besar yang di-*fully fine-tune*.

Keputusan improvement yang diterapkan di notebook ini beserta alasannya:

1. **Loss: BCE+pos_weight polos -> Focal-BCE (pos_weight tetap dipakai, ditambah faktor `(1-p_t)^gamma`).**
   pos_weight exp_001 berentang 0.7 (Support Devices) sampai 26.6 (Pleural Other). pos_weight sebesar itu membuat gradien didominasi sample kelas langka tanpa mempedulikan tingkat kesulitan sample (easy vs hard) - rawan overconfident/noisy pada kelas minor. Focal term meredam kontribusi easy example sehingga training pada kelas imbalance jadi lebih stabil.

2. **Classifier head: 1 linear layer -> 2-layer MLP (`hidden_dim` diisi 256, sebelumnya `None`) + opsi attention pooling.**
   Mean pooling atas `patch_embeddings` membuang informasi spasial, padahal patologi lokal (fracture, pneumothorax kecil) butuh weighting spasial. Attention pooling (weighted sum atas patch token) ditambahkan sebagai opsi (`config['model']['pooling']`), tetap bisa dibalik ke mean pooling untuk pembanding.

3. **Gradual unfreezing backbone** (`freeze_backbone_epochs`), bukan full fine-tune sejak epoch 1.
   Di exp_001, val_loss sudah naik dari epoch 3 padahal head masih baru diinisialisasi acak - backbone ikut berubah sebelum head sempat stabil. Beberapa epoch pertama backbone dibekukan supaya head belajar dulu, baru backbone di-unfreeze bertahap.

4. **Augmentasi ringan** (`use_augmentation`): random rotation kecil (+-5 derajat) + brightness/contrast jitter, **tanpa horizontal flip** (lateralitas CXR bermakna secara klinis, tidak boleh dibalik).
   Menambah regularisasi karena gap train-vs-val macro-F1 di exp_001 cukup lebar (0.55 vs 0.48).

5. **Mixed precision (AMP) + gradient accumulation** (`use_amp`, `grad_accum_steps`).
   Mempercepat training (exp_001 ~33 menit/epoch) dan menstabilkan gradien BCE dengan pos_weight besar pada batch kecil (16) lewat effective batch size yang lebih besar tanpa menambah memori.

6. **Threshold tuning per-pathology**, di samping threshold global (tetap dipertahankan untuk kompatibilitas mundur ke format prompt-injection stage-MEETING).
   Satu threshold global untuk 13 patologi dengan pos_weight 0.7-26.6 secara statistik merugikan kelas minor. **Catatan penting**: threshold per-kelas ini adalah *deviasi* dari desain awal (`pipeline-abstraction-001.md`) yang mengasumsikan satu `threshold` skalar untuk format string `"chexpert findings: {list}"`. Notebook ini menyimpan keduanya (global & per-kelas) - keputusan mana yang dipakai di stage-MEETING perlu didiskusikan ulang, bukan diam-diam diganti.

7. **Logging per-class F1 tiap epoch** (bukan cuma macro-F1 agregat), supaya kolaps di kelas langka terdeteksi lebih awal, tidak tersembunyi di balik angka macro yang didominasi kelas mayoritas.

8. **Bootstrap confidence interval untuk metrik test set.**
   Test set official CheXpert Plus hanya 234 sampel - metrik titik (bal_acc, macro-F1) pada sampel sekecil itu high-variance. CI ditambahkan supaya angka test tidak dibaca sebagai titik pasti.

Semua perubahan konfigurasi ditandai di cell config (section 1) dengan komentar `# [exp_002]` supaya mudah dibedakan dari baseline exp_001.

## 1. Import Library and Setup Configuration

In [ ]:
# import necessary libraries
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.cuda.amp import autocast, GradScaler  # [exp_002] mixed precision
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, balanced_accuracy_score, f1_score
from health_multimodal.image import get_image_inference
from health_multimodal.image.utils import ImageModelType

In [ ]:
# setup reproducibility
def seed_everything(seed=42):
    import random
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [ ]:
# setup configuration
config = {
    "experiment": "exp_002",  # [exp_002] naikkan dari exp_001
    "stage": "stage_b",

   "data": {
        "train_path": r"C:\research-cxr-report_generation\repo\for farhan RRG\train.parquet",
        "dev_path": r"C:\research-cxr-report_generation\repo\for farhan RRG\dev_internal.parquet",
        "test_path": r"C:\research-cxr-report_generation\repo\for farhan RRG\test.parquet",
        "image_col": "actual_image_path",  # Gunakan nama kolom DataFrame
        "label_suffix": "_label"
    },

    "model": {
        "freeze_backbone": False,
        "hidden_dim": 256,          # [exp_002] sebelumnya None (1 linear layer) -> 2-layer MLP
        "num_pathologies": 14,
        "dropout": 0.2,             # [exp_002] naik dari 0.1, head lebih besar butuh regularisasi lebih
        "pooling": "attention"      # [exp_002] "attention" atau "mean" (mean = perilaku exp_001)
    },

    "train": {
        "batch_size": 16,
        "num_epochs": 30,
        "lr": 1.0e-5,
        "backbone_lr_mult": 0.1,          # [exp_002] backbone dilatih lebih pelan daripada head
        "weight_decay": 1.0e-4,
        "grad_clip_norm": 1.0,
        "early_stopping_patience": 5,     # [exp_002] naik dari 3, kasih ruang setelah unfreeze
        "num_workers": 0,
        "seed": 42,
        "lr_scheduler": "ReduceLROnPlateau",
        "lr_scheduler_patience": 3,
        "freeze_backbone_epochs": 2,      # [exp_002] gradual unfreezing: freeze backbone di 2 epoch pertama
        "use_amp": True,                  # [exp_002] mixed precision
        "grad_accum_steps": 2,            # [exp_002] effective batch size = 16 * 2 = 32
        "use_augmentation": True          # [exp_002] rotation + brightness/contrast jitter (train split only)
    },

    "loss": {
        "type": "focal_bce",  # [exp_002] "focal_bce" atau "bce" (bce = perilaku exp_001)
        "gamma": 2.0
    },

    "eval": {
        "bootstrap_n": 1000,    # [exp_002] jumlah resample untuk CI metrik test
        "bootstrap_ci": 0.95
    },

    "output": {
        "checkpoint_dir": r"C:\research-cxr-report_generation\repo\for farhan RRG\rrg_project\checkpoints",
        "log_dir": r"C:\research-cxr-report_generation\repo\for farhan RRG\rrg_project\logs",
        "tuning_dir": r"C:\research-cxr-report_generation\repo\for farhan RRG\rrg_project\tuning",
        "plot_dir": r"C:\research-cxr-report_generation\repo\for farhan RRG\rrg_project\plots",
        "best_model_filename": "stage_b_best.pt"
    },

    "device": "cuda"
}

# create output directories
for key in ["checkpoint_dir", "log_dir", "tuning_dir", "plot_dir"]:
    os.makedirs(config["output"][key], exist_ok=True)

In [ ]:
# define device, if GPU doesn't exist, it'll fallback to CPU usage
device = torch.device(config['device'] if torch.cuda.is_available() else 'cpu')
print(f"using device: {device}")

## 2. Load Data

In [ ]:
# load into a dataframe for each set
df_train = pd.read_parquet(config['data']['train_path'])
df_dev = pd.read_parquet(config['data']['dev_path'])
df_test = pd.read_parquet(config['data']['test_path'])

In [ ]:
# quick exploration

# dataset dimension
print(f"train set: {df_train.shape[0]} rows & {df_train.shape[1]} columns")
print(f"dev set: {df_dev.shape[0]} rows & {df_dev.shape[1]} columns")
print(f"test set: {df_test.shape[0]} rows & {df_test.shape[1]} columns")

# CXR pathology multilabel
label_cols = [col for col in df_train.columns if col.endswith(config['data']['label_suffix'])]
print(f"label columns ({len(label_cols)}): {label_cols}")

# make sure there are 14 CXR pathology labels
assert len(label_cols) == 14, "there have to be 14 CXR pathology labels" 

## 3. Training Utilities

### 3.1 Dataset Class

In [ ]:
# [exp_002] augmentasi ringan khusus train split
# tidak pakai horizontal flip: lateralitas CXR (kiri/kanan) bermakna secara klinis, tidak boleh dibalik
train_augment = transforms.Compose([
    transforms.RandomRotation(degrees=5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1)
])


class CheXpertPlusDataset(Dataset):
    def __init__(self, df, image_col, label_cols, transform=None, augment=None):
        self.df = df.reset_index(drop=True)
        self.image_col = image_col
        self.label_cols = label_cols
        self.transform = transform
        self.augment = augment  # [exp_002] None untuk dev/test, train_augment untuk train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        raw_path = str(row[self.image_col])

        # 1. Perbaiki path folder fisik
        img_path = raw_path.replace(r'preprocessed\PNG', 'image-preprocess')

        try:
            # 2. GUNAKAN 'L' (Grayscale 1-channel):
            # biovilt_transform akan mengubah [1, H, W] menjadi [3, H, W] secara otomatis
            image = Image.open(img_path).convert('L')
        except FileNotFoundError:
            print(f"warning: image not found {img_path}, returning zeros.")
            image = Image.new('L', (448, 448))

        # [exp_002] augmentasi dijalankan sebelum biovilt_transform, di ruang PIL
        if self.augment is not None:
            image = self.augment(image)

        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(row[self.label_cols].values.astype(np.float32), dtype=torch.float32)
        return image, labels


### 3.2 Load BioViL-T Image Encoder

In [ ]:
# load biovil-t image encoder
image_engine = get_image_inference(ImageModelType.BIOVIL_T)
biovilt_transform = image_engine.transform
bio_model = image_engine.model
bio_model.to(device)
bio_model.eval()

print("biovil-t loaded successfully")

In [ ]:
# detect vision hidden dimension

sample_row = df_train.iloc[0]
sample_img_path = sample_row[config['data']['image_col']]
try:
    sample_img = Image.open(sample_img_path).convert('L')
except FileNotFoundError:
    sample_img = Image.new('L', (224, 224))

sample_pixel_values = biovilt_transform(sample_img).unsqueeze(0).to(device)

with torch.no_grad():
    dummy_out = bio_model(sample_pixel_values)

    # extract features tensor from dummy_out
    if hasattr(dummy_out, 'patch_embeddings'):
        features = dummy_out.patch_embeddings
        print("using patch_embeddings")
    elif hasattr(dummy_out, 'last_hidden_state'):
        features = dummy_out.last_hidden_state
        print("using last_hidden_state")
    else:
        # if dummy_out is a tensor directly, use it
        if isinstance(dummy_out, torch.Tensor):
            features = dummy_out
            print("using raw tensor output")
        else:
            # fallback: try to get encoder output
            try:
                features = bio_model.encoder(sample_pixel_values)
                print("using encoder output")
            except:
                raise RuntimeError("Cannot extract features from BioViL-T output")

    # determine vision dimension based on features shape
    if features.dim() == 3:
        # (B, seq, D)
        vision_dim = features.shape[-1]
    elif features.dim() == 4:
        # (B, C, H, W)
        vision_dim = features.shape[1]
    else:
        # fallback
        vision_dim = features.shape[-1]

    # sanity check: if vision_dim is suspiciously small (like 14), set to a known good value
    if vision_dim < 100:
        print(f"warning: detected vision_dim={vision_dim}, which is suspiciously small. forcing to 768 (default BioViL-T dimension).")
        vision_dim = 768

config['vision_dim'] = vision_dim
print(f"vision hidden dimension: {vision_dim}")

### 3.3 Create Dataset & Dataloader

In [ ]:
# [exp_002] augment hanya dipasang di train_dataset, dev/test tetap deterministik
active_train_augment = train_augment if config['train']['use_augmentation'] else None

train_dataset = CheXpertPlusDataset(df_train, config['data']['image_col'], label_cols, transform=biovilt_transform, augment=active_train_augment)
dev_dataset = CheXpertPlusDataset(df_dev, config['data']['image_col'], label_cols, transform=biovilt_transform)
test_dataset = CheXpertPlusDataset(df_test, config['data']['image_col'], label_cols, transform=biovilt_transform)

train_loader = DataLoader(train_dataset, batch_size=config['train']['batch_size'], shuffle=True, num_workers=config['train']['num_workers'], pin_memory=True)
dev_loader = DataLoader(dev_dataset, batch_size=config['train']['batch_size'], shuffle=False, num_workers=config['train']['num_workers'], pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=config['train']['batch_size'], shuffle=False, num_workers=config['train']['num_workers'], pin_memory=True)

print(f"train batches: {len(train_loader)}, dev batches: {len(dev_loader)}, test batches: {len(test_loader)}")

### 3.4 Compute Weights to Mitigate Imbalanced Class

In [ ]:
def compute_pos_weights(df: pd.DataFrame, label_cols: list) -> torch.Tensor:
    """
    per-pathology pos_weight for nn.BCEWithLogitsLoss, from train prevalence only.
    pos_weight_i = num_negative_i / num_positive_i.
    """
    pos_counts = (df[label_cols] == 1.0).sum()
    neg_counts = (df[label_cols] == 0.0).sum()
    pos_weight = (neg_counts / pos_counts.clip(lower=1)).to_numpy(dtype="float32")
    return torch.tensor(pos_weight)

pos_weight = compute_pos_weights(df_train, label_cols).to(device)

print("class weights (pos_weight = N_neg / N_pos):")
for col, w in zip(label_cols, pos_weight.cpu().numpy()):
    print(f"  {col}: {w:.3f}")

In [ ]:
# [exp_002] focal loss dikombinasikan dengan pos_weight yang sudah ada
# alasan: pos_weight besar (sampai 26.6) pada BCE murni bikin gradien didominasi kelas langka
# tanpa mempedulikan tingkat kesulitan sample -> focal term meredam easy example, training lebih stabil
class FocalBCEWithLogitsLoss(nn.Module):
    def __init__(self, pos_weight, gamma=2.0):
        super().__init__()
        self.pos_weight = pos_weight
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction='none'
        )
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        focal_term = (1 - p_t).clamp(min=1e-6) ** self.gamma
        return (focal_term * bce).mean()


if config['loss']['type'] == 'focal_bce':
    criterion = FocalBCEWithLogitsLoss(pos_weight=pos_weight, gamma=config['loss']['gamma'])
    print(f"using focal-bce loss (gamma={config['loss']['gamma']})")
else:
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    print("using plain bce loss (exp_001 baseline)")

### 3.5 Define Stage-B Classifier Model

In [ ]:
class StageBClassifier(nn.Module):
    def __init__(self, config, backbone, vision_dim, freeze_backbone=False):
        super().__init__()
        self.backbone = backbone
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        self.hidden_dim = config['model']['hidden_dim']
        self.num_classes = config['model']['num_pathologies']
        self.dropout_rate = config['model']['dropout']
        self.pooling = config['model']['pooling']  # [exp_002] "attention" atau "mean"

        in_features = vision_dim

        # [exp_002] attention pooling: 1 learnable query yang menghitung bobot atas patch token
        if self.pooling == "attention":
            self.attn_pool = nn.Linear(in_features, 1)

        if self.hidden_dim is not None:
            self.classifier = nn.Sequential(
                nn.Dropout(self.dropout_rate),
                nn.Linear(in_features, self.hidden_dim),
                nn.ReLU(),
                nn.Dropout(self.dropout_rate),
                nn.Linear(self.hidden_dim, self.num_classes)
            )
        else:
            self.classifier = nn.Sequential(
                nn.Dropout(self.dropout_rate),
                nn.Linear(in_features, self.num_classes)
            )

    def set_backbone_trainable(self, trainable: bool):
        # [exp_002] dipakai untuk gradual unfreezing di training loop
        for param in self.backbone.parameters():
            param.requires_grad = trainable

    def forward(self, x):
        out = self.backbone(x)

        # extract features
        if hasattr(out, 'patch_embeddings'):
            features = out.patch_embeddings
        elif hasattr(out, 'last_hidden_state'):
            features = out.last_hidden_state
        else:
            features = out

        # ensure features is a tensor
        if not isinstance(features, torch.Tensor):
            raise TypeError(f"Expected Tensor, got {type(features)}")

        # pooling: reduce to (B, D)
        if features.dim() == 3:
            # (B, seq, D)
            if self.pooling == "attention":
                attn_scores = self.attn_pool(features)              # (B, seq, 1)
                attn_weights = torch.softmax(attn_scores, dim=1)    # (B, seq, 1)
                pooled = (features * attn_weights).sum(dim=1)       # (B, D)
            else:
                pooled = features.mean(dim=1)  # (B, seq, D) -> (B, D)
        elif features.dim() == 4:
            if self.pooling == "attention":
                b, c, h, w = features.shape
                flat = features.flatten(2).transpose(1, 2)          # (B, H*W, C)
                attn_scores = self.attn_pool(flat)                  # (B, H*W, 1)
                attn_weights = torch.softmax(attn_scores, dim=1)
                pooled = (flat * attn_weights).sum(dim=1)            # (B, C)
            else:
                pooled = features.mean(dim=[2, 3])  # (B, C, H, W) -> (B, C)
        else:
            # already pooled? just flatten if needed
            pooled = features.flatten(1) if features.dim() > 2 else features

        logits = self.classifier(pooled)
        return logits

# instantiate model
model = StageBClassifier(
    config,
    backbone=bio_model,
    vision_dim=config['vision_dim'],
    freeze_backbone=config['model']['freeze_backbone']
)
model = model.to(device)

print(f"model initialized on {device}")
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"total parameters: {total_params/1e6:.2f}M, trainable: {trainable_params/1e6:.2f}M")

## 4. Train Loop

In [ ]:
no_finding_idx = None
for i, col in enumerate(label_cols):
    if col.lower().startswith('no finding'):
        no_finding_idx = i
        break
if no_finding_idx is None:
    no_finding_idx = 0
other_indices = [i for i in range(len(label_cols)) if i != no_finding_idx]

print(f"no finding index: {no_finding_idx}")
print(f"other indices (13 pathologies): {other_indices}")

In [ ]:
torch.cuda.empty_cache()

In [ ]:
# [exp_002] gradual unfreezing: backbone dibekukan di N epoch pertama supaya head stabil dulu
model.set_backbone_trainable(False)
backbone_frozen = True
print(f"backbone frozen for the first {config['train']['freeze_backbone_epochs']} epoch(s)")

In [ ]:
# [exp_002] differential learning rate: backbone dilatih lebih pelan dari head
head_params = list(model.classifier.parameters())
if config['model']['pooling'] == "attention":
    head_params += list(model.attn_pool.parameters())
backbone_params = list(model.backbone.parameters())

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": config['train']['lr'] * config['train']['backbone_lr_mult']},
        {"params": head_params, "lr": config['train']['lr']}
    ],
    weight_decay=config['train']['weight_decay']
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=config['train']['lr_scheduler_patience'], factor=0.5
)

# [exp_002] amp scaler
scaler = GradScaler(enabled=config['train']['use_amp'])

# history tracking
early_stop_counter = 0
best_val_macro_f1 = 0.0
train_losses, val_losses = [], []
train_macro_f1s, val_macro_f1s = [], []
val_per_class_f1_history = []  # [exp_002] simpan f1 per kelas tiap epoch, bukan cuma macro


print("starting training (optimizer: adamw, early stopping based on macro-f1 on 13 pathologies, excluding no finding)")
for epoch in range(1, config['train']['num_epochs'] + 1):

    # [exp_002] unfreeze backbone setelah freeze_backbone_epochs terlewati
    if backbone_frozen and epoch > config['train']['freeze_backbone_epochs']:
        model.set_backbone_trainable(True)
        backbone_frozen = False
        print(f"epoch {epoch}: backbone unfrozen, full fine-tuning starts now")

    # training phase
    model.train()
    train_loss = 0.0
    train_preds, train_labels = [], []
    optimizer.zero_grad()
    for step, (images, labels) in enumerate(tqdm(train_loader, desc=f"epoch {epoch}/{config['train']['num_epochs']} (train)")):
        images, labels = images.to(device), labels.to(device)

        with autocast(enabled=config['train']['use_amp']):
            logits = model(images)
            loss = criterion(logits, labels) / config['train']['grad_accum_steps']

        scaler.scale(loss).backward()

        # [exp_002] gradient accumulation: step optimizer tiap grad_accum_steps batch
        if (step + 1) % config['train']['grad_accum_steps'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config['train']['grad_clip_norm'])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_loss += loss.item() * config['train']['grad_accum_steps'] * images.size(0)

        probs = torch.sigmoid(logits)
        train_preds.append(probs.detach().cpu().numpy())
        train_labels.append(labels.detach().cpu().numpy())
    train_loss /= len(train_loader.dataset)
    train_losses.append(train_loss)

    # compute train macro-f1 on 13 pathologies (excluding no finding)
    train_preds = np.vstack(train_preds)
    train_labels = np.vstack(train_labels)
    train_preds_binary = (train_preds > 0.5).astype(int)
    train_preds_other = train_preds_binary[:, other_indices]
    train_labels_other = train_labels[:, other_indices]
    train_macro_f1 = f1_score(train_labels_other, train_preds_other, average='macro', zero_division=0)
    train_macro_f1s.append(train_macro_f1)

    # validation phase
    model.eval()
    val_loss = 0.0
    val_preds, val_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(dev_loader, desc=f"epoch {epoch}/{config['train']['num_epochs']} (val)"):
            images, labels = images.to(device), labels.to(device)
            with autocast(enabled=config['train']['use_amp']):
                logits = model(images)
                loss = criterion(logits, labels)
            val_loss += loss.item() * images.size(0)
            probs = torch.sigmoid(logits)
            val_preds.append(probs.cpu().numpy())
            val_labels.append(labels.cpu().numpy())
    val_loss /= len(dev_loader.dataset)
    val_losses.append(val_loss)

    # compute val macro-f1 on 13 pathologies (excluding no finding)
    val_preds = np.vstack(val_preds)
    val_labels = np.vstack(val_labels)
    val_preds_binary = (val_preds > 0.5).astype(int)
    val_preds_other = val_preds_binary[:, other_indices]
    val_labels_other = val_labels[:, other_indices]
    val_macro_f1 = f1_score(val_labels_other, val_preds_other, average='macro', zero_division=0)
    val_macro_f1s.append(val_macro_f1)

    # [exp_002] log per-class f1 tiap epoch, supaya kolaps kelas langka kelihatan lebih awal
    per_class_f1 = f1_score(val_labels_other, val_preds_other, average=None, zero_division=0)
    val_per_class_f1_history.append(per_class_f1)
    worst_idx = np.argsort(per_class_f1)[:3]
    worst_report = ", ".join(f"{label_cols[other_indices[i]]}={per_class_f1[i]:.3f}" for i in worst_idx)

    print(f"epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, train_macro_f1={train_macro_f1:.4f}, val_macro_f1={val_macro_f1:.4f}")
    print(f"  -> 3 kelas terlemah (val f1): {worst_report}")

    # scheduler step based on val loss
    scheduler.step(val_loss)

    # early stopping & checkpoint based on val macro-f1 (excluding no finding)
    if val_macro_f1 > best_val_macro_f1:
        best_val_macro_f1 = val_macro_f1
        early_stop_counter = 0
        best_model_path = os.path.join(config['output']['checkpoint_dir'], config['output']['best_model_filename'])
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_macro_f1': best_val_macro_f1,
        }, best_model_path)
        print(f"  -> new best model saved (val_macro_f1={val_macro_f1:.4f})")
    else:
        early_stop_counter += 1
        if early_stop_counter >= config['train']['early_stopping_patience']:
            print(f"early stopping triggered at epoch {epoch} (no improvement for {config['train']['early_stopping_patience']} epochs)")
            break

print(f"\ntraining finished. best validation macro-f1 (13 pathologies, excluding no finding): {best_val_macro_f1:.4f}")

## 5. Model Evaluation

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='train loss')
plt.plot(val_losses, label='val loss')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.title('training & validation loss (focal-bce)')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(train_macro_f1s, label='train macro-f1 (13 pathologies)')
plt.plot(val_macro_f1s, label='val macro-f1 (13 pathologies)')
plt.xlabel('epoch')
plt.ylabel('macro-f1')
plt.legend()
plt.title('training & validation macro-f1 (excluding no finding)')
plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(config['output']['plot_dir'], 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# [exp_002] heatmap per-class f1 sepanjang epoch, untuk lihat kelas mana yang kolaps/plateau lebih awal
per_class_f1_matrix = np.vstack(val_per_class_f1_history)  # (n_epoch, 13)
other_label_names = [label_cols[i] for i in other_indices]

plt.figure(figsize=(10, 6))
sns.heatmap(
    per_class_f1_matrix.T,
    xticklabels=range(1, len(val_per_class_f1_history) + 1),
    yticklabels=other_label_names,
    cmap='viridis',
    annot=False
)
plt.xlabel('epoch')
plt.title('val f1 per pathology sepanjang epoch (excluding no finding)')
plt.tight_layout()
plt.savefig(os.path.join(config['output']['plot_dir'], 'per_class_f1_history.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
best_model_path = os.path.join(config['output']['checkpoint_dir'], config['output']['best_model_filename'])
model.load_state_dict(torch.load(best_model_path)['model_state_dict'])
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="evaluating test set"):
        images = images.to(device)
        logits = model(images)
        probs = torch.sigmoid(logits)
        test_preds.append(probs.cpu().numpy())
        test_labels.append(labels.cpu().numpy())

test_preds = np.vstack(test_preds)
test_labels = np.vstack(test_labels)
test_preds_binary = (test_preds > 0.5).astype(int)


print("test set performance (threshold=0.5)")
print(classification_report(test_labels, test_preds_binary, target_names=label_cols, zero_division=0))

bal_accs = [
    balanced_accuracy_score(test_labels[:, i], test_preds_binary[:, i])
    for i in range(test_labels.shape[1])
]
test_bal_acc = np.mean(bal_accs)

print(f"\nbalanced accuracy on test set: {test_bal_acc:.4f}")

In [ ]:
# [exp_002] bootstrap CI untuk bal_acc test set, karena n=234 kecil dan rawan noise
def bootstrap_bal_acc_ci(labels, preds_binary, n_boot=1000, ci=0.95, seed=42):
    rng = np.random.default_rng(seed)
    n_samples = labels.shape[0]
    boot_scores = []
    for _ in range(n_boot):
        idx = rng.integers(0, n_samples, size=n_samples)
        boot_labels = labels[idx]
        boot_preds = preds_binary[idx]
        per_class = [
            balanced_accuracy_score(boot_labels[:, i], boot_preds[:, i])
            for i in range(boot_labels.shape[1])
            if len(np.unique(boot_labels[:, i])) > 1  # skip kelas yang kebetulan single-class di resample
        ]
        if len(per_class) > 0:
            boot_scores.append(np.mean(per_class))
    lower = np.percentile(boot_scores, (1 - ci) / 2 * 100)
    upper = np.percentile(boot_scores, (1 + ci) / 2 * 100)
    return np.mean(boot_scores), lower, upper


boot_mean, boot_lo, boot_hi = bootstrap_bal_acc_ci(
    test_labels, test_preds_binary,
    n_boot=config['eval']['bootstrap_n'], ci=config['eval']['bootstrap_ci']
)
print(f"bal_acc bootstrap ({config['eval']['bootstrap_n']} resample, n_test={test_labels.shape[0]}):")
print(f"  mean={boot_mean:.4f}, {int(config['eval']['bootstrap_ci']*100)}% CI=[{boot_lo:.4f}, {boot_hi:.4f}]")

## 6. Threshold Tuning

In [ ]:
dev_probs, dev_labels = [], []
model.eval()
with torch.no_grad():
    for images, labels in tqdm(dev_loader, desc="extracting dev probabilities"):
        images = images.to(device)
        logits = model(images)
        probs = torch.sigmoid(logits)
        dev_probs.append(probs.cpu().numpy())
        dev_labels.append(labels.cpu().numpy())

dev_probs = np.vstack(dev_probs)
dev_labels = np.vstack(dev_labels)

thresholds = np.arange(0.1, 0.91, 0.05)
best_th = 0.5
best_macro_f1 = 0.0

print("threshold tuning - global (macro f1 on dev set, excluding no finding)")
for th in thresholds:
    preds_th = (dev_probs[:, other_indices] >= th).astype(int)
    labels_th = dev_labels[:, other_indices]
    macro_f1 = f1_score(labels_th, preds_th, average='macro', zero_division=0)
    print(f"threshold={th:.2f}: macro_f1={macro_f1:.4f}")
    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_th = th

print(f"\nbest global threshold: {best_th:.2f} (macro f1 on dev: {best_macro_f1:.4f})")

# save best threshold (dipertahankan untuk kompatibilitas mundur dengan format prompt-injection stage-MEETING)
with open(os.path.join(config['output']['tuning_dir'], 'best_threshold.txt'), 'w') as f:
    f.write(str(best_th))

# save dev probabilities for future use
np.save(os.path.join(config['output']['tuning_dir'], 'dev_probs.npy'), dev_probs)
np.save(os.path.join(config['output']['tuning_dir'], 'dev_labels.npy'), dev_labels)

In [ ]:
# [exp_002] threshold tuning per-pathology
# CATATAN DESAIN: pipeline-abstraction-001.md mengasumsikan satu `threshold` skalar untuk format
# prompt-injection stage-MEETING ("chexpert findings: {list}"). threshold per-kelas di bawah ini
# adalah opsi analisis tambahan, bukan pengganti otomatis -> keputusan mana yang dipakai di
# stage-MEETING perlu didiskusikan/didokumentasikan ulang, tidak diam-diam menggantikan best_th.
per_class_best_th = {}
per_class_best_f1 = {}

print("threshold tuning - per pathology (f1 on dev set)")
for local_i, global_i in enumerate(other_indices):
    col_name = label_cols[global_i]
    col_probs = dev_probs[:, global_i]
    col_labels = dev_labels[:, global_i]

    best_col_th, best_col_f1 = 0.5, 0.0
    for th in thresholds:
        preds_th = (col_probs >= th).astype(int)
        f1 = f1_score(col_labels, preds_th, zero_division=0)
        if f1 > best_col_f1:
            best_col_f1 = f1
            best_col_th = th

    per_class_best_th[col_name] = float(best_col_th)
    per_class_best_f1[col_name] = float(best_col_f1)
    print(f"  {col_name}: best_threshold={best_col_th:.2f}, f1={best_col_f1:.4f}")

macro_f1_per_class_th = np.mean(list(per_class_best_f1.values()))
print(f"\nmacro-f1 kalau tiap kelas pakai threshold optimalnya sendiri: {macro_f1_per_class_th:.4f}")
print(f"macro-f1 dengan threshold global ({best_th:.2f})            : {best_macro_f1:.4f}")

# save per-class threshold
import json
with open(os.path.join(config['output']['tuning_dir'], 'best_threshold_per_class.json'), 'w') as f:
    json.dump(per_class_best_th, f, indent=2)

In [ ]:
truly_normal_mask = df_dev['No Finding_label'] == 1.0
truly_normal_probs = dev_probs[truly_normal_mask]

correctly_flagged_normal = 0
total_normal = len(truly_normal_probs)

for row_probs in truly_normal_probs:
    positive_others = [i for i in other_indices if row_probs[i] >= best_th]
    if len(positive_others) == 0:
        correctly_flagged_normal += 1

normal_detection_rate = correctly_flagged_normal / total_normal if total_normal > 0 else 0.0


print("validation on genuinely normal cases (dev set), threshold global:")
print(f"total genuinely normal samples in dev: {total_normal}")
print(f"correctly flagged as 'no significant findings': {correctly_flagged_normal}")
print(f"normal detection rate: {normal_detection_rate:.2%}")

# abnormal recall rate (sanity check untuk memastikan threshold tidak terlalu tinggi)
abnormal_mask = (df_dev[label_cols].drop(columns=['No Finding_label']).sum(axis=1) > 0)
abnormal_probs = dev_probs[abnormal_mask]
abnormal_detected = 0
total_abnormal = len(abnormal_probs)

for row_probs in abnormal_probs:
    positive_others = [i for i in other_indices if row_probs[i] >= best_th]
    if len(positive_others) > 0:
        abnormal_detected += 1

abnormal_recall_rate = abnormal_detected / total_abnormal if total_abnormal > 0 else 0.0
print(f"total genuinely abnormal samples in dev: {total_abnormal}")
print(f"correctly flagged with at least one pathology: {abnormal_detected}")
print(f"abnormal recall rate: {abnormal_recall_rate:.2%}")

print("\ninterpretation:")
if normal_detection_rate < 0.90:
    print("  -> warning: threshold too low. many normal cases are being over-called as abnormal.")
elif normal_detection_rate > 0.99 and abnormal_recall_rate < 0.50:
    print("  -> warning: threshold too high. many abnormal cases are being missed.")
else:
    print("  -> threshold balanced well between preserving normal cases and detecting abnormalities.")

In [ ]:
# evaluate on test set with optimal global threshold
test_preds_optimal = (test_preds[:, other_indices] >= best_th).astype(int)
test_labels_optimal = test_labels[:, other_indices]

print(f"test set performance (optimal global threshold={best_th:.2f})")
print(classification_report(test_labels_optimal, test_preds_optimal, target_names=[label_cols[i] for i in other_indices], zero_division=0))

test_bal_acc_optimal = balanced_accuracy_score(test_labels_optimal, test_preds_optimal)
print(f"\nbalanced accuracy on test set (optimal global threshold): {test_bal_acc_optimal:.4f}")

In [ ]:
# [exp_002] evaluate on test set with per-class optimal threshold, untuk pembanding
test_preds_per_class_th = np.zeros_like(test_labels_optimal)
for local_i, global_i in enumerate(other_indices):
    col_name = label_cols[global_i]
    th = per_class_best_th[col_name]
    test_preds_per_class_th[:, local_i] = (test_preds[:, global_i] >= th).astype(int)

print("test set performance (per-class optimal threshold)")
print(classification_report(test_labels_optimal, test_preds_per_class_th, target_names=[label_cols[i] for i in other_indices], zero_division=0))

test_bal_acc_per_class_th = balanced_accuracy_score(test_labels_optimal, test_preds_per_class_th)
print(f"\nbalanced accuracy on test set (per-class threshold): {test_bal_acc_per_class_th:.4f}")

## 7. Summary of Stage-B

In [ ]:
summary = f"""
====================================================================
stage-b training summary
====================================================================
experiment: {config['experiment']}
stage: {config['stage']}
device: {device}

dataset:
  - train: {len(df_train)} samples
  - dev: {len(df_dev)} samples
  - test: {len(df_test)} samples

model:
  - backbone: biovil-t
  - vision dimension: {config['vision_dim']}
  - pooling: {config['model']['pooling']}
  - hidden dim: {config['model']['hidden_dim']}
  - dropout: {config['model']['dropout']}
  - total params: {total_params/1e6:.2f}m
  - trainable params: {trainable_params/1e6:.2f}m

training:
  - loss: {config['loss']['type']} (gamma={config['loss']['gamma']})
  - batch size: {config['train']['batch_size']} (effective: {config['train']['batch_size'] * config['train']['grad_accum_steps']})
  - learning rate (head / backbone): {config['train']['lr']} / {config['train']['lr'] * config['train']['backbone_lr_mult']}
  - weight decay: {config['train']['weight_decay']}
  - freeze_backbone_epochs: {config['train']['freeze_backbone_epochs']}
  - use_amp: {config['train']['use_amp']}
  - use_augmentation: {config['train']['use_augmentation']}
  - epochs: {len(train_losses)}
  - best validation macro-f1 (13 pathologies, excluding no finding): {best_val_macro_f1:.4f}

threshold tuning (on dev set, excluding no finding):
  - best global threshold: {best_th:.2f}
  - macro f1 at best global threshold: {best_macro_f1:.4f}
  - macro f1 with per-class thresholds: {macro_f1_per_class_th:.4f}

normal detection rate (dev set, genuinely normal cases, global threshold):
  - correctly flagged as normal: {correctly_flagged_normal}/{total_normal}
  - normal detection rate: {normal_detection_rate:.2%}
  - abnormal recall rate: {abnormal_recall_rate:.2%}

test performance:
  - bal_acc (threshold=0.5): {test_bal_acc:.4f}
  - bal_acc (optimal global threshold={best_th:.2f}): {test_bal_acc_optimal:.4f}
  - bal_acc (per-class threshold): {test_bal_acc_per_class_th:.4f}
  - bal_acc bootstrap {int(config['eval']['bootstrap_ci']*100)}% CI (threshold=0.5): [{boot_lo:.4f}, {boot_hi:.4f}] (mean={boot_mean:.4f}, n_boot={config['eval']['bootstrap_n']})

output directory: {config['output']['checkpoint_dir']}
====================================================================
"""

print(summary)

with open(os.path.join(config['output']['log_dir'], 'training_summary.txt'), 'w') as f:
    f.write(summary)

print("\nall done. stage-b training and evaluation complete (exp_002).")